### MF-RMF Computation and Saving


In [1]:
import numpy as np
import h5py
from typing import Tuple, Optional
from loguru import logger
import numpy as np
import itertools
from typing import Union
import numpy as np
import pickle

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

In [3]:
def hdf5_data_load(data_path: str, data_file_name: str, data_type: str) -> (
        Tuple)[Optional[np.ndarray], Optional[np.ndarray]]:
    """
    Load dataset from an HDF5 file based on the specified data type.

    Parameters:
        data_path (str): The path to the directory containing the HDF5 file.
        data_file_name (str): The name of the HDF5 file.
        data_type (str): The type of dataset to load ('Train' or 'Test').

    Returns:
        Tuple[Optional[np.ndarray], Optional[np.ndarray]]: A tuple containing the feature
        matrix (X) and labels (y), or (None, None) if an error occurs.
    """

    # Dictionary to map data types to dataset keys in the HDF5 file
    data_keys = {
        "Train": ("X_train", "y_train"),
        "Test": ("X_test", "y_test")
    }

    # Retrieve appropriate dataset keys based on specified data_type
    dataset_keys = data_keys.get(data_type)

    # If the data_type is not recognized, return None
    if not dataset_keys:
        logger.error(f"Data type '{data_type}' is not supported. Choose 'Train' or 'Test'.")
        return None, None

    # Using a context manager to handle the file
    with h5py.File(f"{data_path}/{data_file_name}", 'r') as hf:
        X = np.array(hf[dataset_keys[0]])
        y = np.array(hf[dataset_keys[1]])
    return X, y

In [4]:
def find_ground_excited_pairs(X, y, qubit):
    """
    Identify ground and excited state traces for a specified qubit position.

    Parameters:
    - X: Array of traces, shape (num_traces, trace_length, 2)
    - y: Array of binary string labels, shape (num_traces,)
    - qubit: Position of the qubit (1 to 5)

    Returns:
    - ground_state_traces: Array of ground state traces
    - excited_state_traces: Array of excited state traces
    - ground_state_labels: Array of labels for ground state traces
    - excited_state_labels: Array of labels for excited state traces
    """
    # Validate qubit position
    if qubit < 1 or qubit > 5:
        raise ValueError("qubit position should be between 1 and 5.")
    
    # Adjust the qubit position to match the reversed order (Q5 -> 0, Q4 -> 1, ..., Q1 -> 4)
    qubit_position = 5 - qubit

    # Step 1: Initialize lists for ground and excited state traces and labels
    ground_state_traces = []
    excited_state_traces = []
    ground_state_labels = []
    excited_state_labels = []

    # Step 2: Iterate over unique binary labels in y
    unique_labels = set(y)
    
    for label in unique_labels:
        # Check if the specified qubit position is excited ('1') in this label
        if label[qubit_position] == '1':
            # Find the ground state counterpart by setting the qubit position to '0'
            counterpart_binary = list(label)
            counterpart_binary[qubit_position] = '0'
            counterpart_label = "".join(counterpart_binary)

            # Select traces from X based on matching labels
            excited_state_indices = np.where(y == label)[0]  # Indices for excited state
            ground_state_indices = np.where(y == counterpart_label)[0]  # Indices for ground state

            # Append the selected traces and their labels
            excited_state_traces.append(X[excited_state_indices])
            ground_state_traces.append(X[ground_state_indices])
            excited_state_labels.append(y[excited_state_indices])
            ground_state_labels.append(y[ground_state_indices])

    # Convert lists to arrays and concatenate to ensure consistency in shape
    return (
        np.concatenate(ground_state_traces), 
        np.concatenate(excited_state_traces), 
        np.concatenate(ground_state_labels), 
        np.concatenate(excited_state_labels)
    )

In [5]:
def get_relaxation_traces(ground_state_traces, excited_state_traces, approach="elementwise"):
    """
    Identifies relaxation traces using different approaches based on the specified method.
    
    Parameters:
    - ground_state_traces: List or array of ground state traces, shape (N, 500, 2)
    - excited_state_traces: List or array of excited state traces, shape (N, 500, 2)
    - approach: String, either "average" for Approach 1 or "mean_point" for Approach 2
    
    Returns:
    - Trelax: List of relaxation traces
    - Tground: List of corresponding ground state traces
    """
    mean0 = []  # Will store means of ground state traces
    mean1 = []  # Will store means of excited state traces

    # Calculate means for I and Q components separately for both ground and excited states
    for i in range(len(ground_state_traces)):
        mean0.append([np.mean(ground_state_traces[i][..., 0]), np.mean(ground_state_traces[i][..., 1])])
        mean1.append([np.mean(excited_state_traces[i][..., 0]), np.mean(excited_state_traces[i][..., 1])])

    # Calculate centroids
    centroid0 = np.mean(np.array(mean0), axis=0)
    centroid1 = np.mean(np.array(mean1), axis=0)

    # Calculate radius
    radius = np.linalg.norm(centroid0 - centroid1) / 2

    Trelax = []
    Tground = []

    for i, tr1 in enumerate(excited_state_traces):
        # Calculate the mean I and Q components for tr1
        tr1_mean = np.mean(tr1, axis=0)
        # Calculate Euclidean distance between the mean of tr1 and centroid0
        distance_to_centroid = np.linalg.norm(tr1_mean - centroid0)
        if distance_to_centroid <= radius:
            Trelax.append(excited_state_traces[i])
            Tground.append(ground_state_traces[i])


    return (
        np.array(Trelax), 
        np.array(Tground)
    )

In [6]:
def compute_mf_envelopes(ground_state_traces, excited_state_traces):
    """
    Compute the matched filter envelopes for the I and Q components based on ground and excited state traces.

    Parameters:
    - ground_state_traces: Array of ground state traces, shape (num_traces, 500, 2)
    - excited_state_traces: Array of excited state traces, shape (num_traces, 500, 2)

    Returns:
    - mf_envelope_I: The matched filter envelope for the I component
    - mf_envelope_Q: The matched filter envelope for the Q component
    """
    # # Separate I and Q components for ground and excited state traces
    ground_I, ground_Q = ground_state_traces[..., 0], ground_state_traces[..., 1]  # Shape: (num_traces, 500)
    excited_I, excited_Q = excited_state_traces[..., 0], excited_state_traces[..., 1]  # Shape: (num_traces, 500)
    
    # Calculate mean differences for I and Q components across time bins
    mean_diff_I = np.mean(ground_I - excited_I, axis=0)  # Shape: (500,)
    mean_diff_Q = np.mean(ground_Q - excited_Q, axis=0)  # Shape: (500,)

    # Calculate variances for I and Q components across time bins
    var_diff_I = np.var(ground_I - excited_I, axis=0) # Shape: (500,)
    var_diff_Q = np.var(ground_Q - excited_Q, axis=0)   # Shape: (500,)

    #####

    epsilon = 1e-10  # A small value to prevent division by zero
    mf_envelope_I = mean_diff_I / (var_diff_I + epsilon)
    mf_envelope_Q = mean_diff_Q / (var_diff_Q + epsilon)
        
    # # Compute the matched filter envelopes
    # mf_envelope_I = mean_diff_I / var_diff_I  # Shape: (500,)
    # mf_envelope_Q = mean_diff_Q / var_diff_Q  # Shape: (500,)

    return mf_envelope_I, mf_envelope_Q

def compute_rmf_envelopes(ground_state_traces, relaxed_state_traces):
    """
    Compute the matched filter envelopes for the I and Q components based on ground and excited state traces.

    Parameters:
    - ground_state_traces: Array of ground state traces, shape (num_traces, 500, 2)
    - excited_state_traces: Array of excited state traces, shape (num_traces, 500, 2)

    Returns:
    - mf_envelope_I: The matched filter envelope for the I component
    - mf_envelope_Q: The matched filter envelope for the Q component
    """
    # Separate I and Q components for ground and excited state traces
    ground_I, ground_Q = ground_state_traces[..., 0], ground_state_traces[..., 1]  # Shape: (num_traces, 500)
    relaxed_I, relaxed_Q = relaxed_state_traces[..., 0], relaxed_state_traces[..., 1]  # Shape: (num_traces, 500)
    
    # Compute mean and variance differences for I and Q components
    mean_diff_I = np.mean(relaxed_I - ground_I, axis=0)  # Shape: (500,)
    var_diff_I = np.var(relaxed_I - ground_I, axis=0)     # Shape: (500,)
    
    mean_diff_Q = np.mean(relaxed_Q - ground_Q, axis=0)  # Shape: (500,)
    var_diff_Q = np.var(relaxed_Q - ground_Q, axis=0)    # Shape: (500,)
    
    # Compute the matched filter envelopes

    
    epsilon = 1e-10  # A small value to prevent division by zero
    rmf_envelope_I = mean_diff_I / (var_diff_I + epsilon)
    rmf_envelope_Q = mean_diff_Q / (var_diff_Q + epsilon)
    
    # rmf_envelope_I = mean_diff_I / var_diff_I  # Shape: (500,)
    # rmf_envelope_Q = mean_diff_Q / var_diff_Q  # Shape: (500,)

    return rmf_envelope_I, rmf_envelope_Q



In [7]:
def apply_mf_rmf(traces, mf_rmf_envelope_I, mf_rmf_envelope_Q):
    """
    Apply the matched filter to a batch of traces.

    Parameters:
    - traces: The batch of traces to which the MF should be applied (shape: [num_traces, 500, 2])
    - mf_rmf_envelope_I: The matched filter or RMF envelope for the I component (shape: [500])
    - mf_rmf_envelope_Q: The matched filter or RMF envelope for the Q component (shape: [500])

    Returns:
    - mf_outputs: The output of the matched filter for each trace in the batch (shape: [num_traces])
    """
    # Separate I and Q components from the batch of traces
    I_traces = traces[..., 0]  # Shape: (num_traces, 500)
    Q_traces = traces[..., 1]  # Shape: (num_traces, 500)
    
    # Apply the MF envelope by taking the dot product across each trace in the batch
    mf_rmf_output_I = np.dot(I_traces, mf_rmf_envelope_I)  # Shape: (num_traces,)
    mf_rmf_output_Q = np.dot(Q_traces, mf_rmf_envelope_Q)  # Shape: (num_traces,)
    
    # Final MF output is the sum of the I and Q component outputs
    mf_rmf_outputs = mf_rmf_output_I + mf_rmf_output_Q  # Shape: (num_traces,)
    
    return mf_rmf_outputs

In [8]:
def normalize_data(X_train, X_test):
    X_mean = np.mean(X_train, axis=0)
    X_std = np.std(X_train, axis=0)

    X_train_norm = (X_train - X_mean) / X_std
    X_test_norm = (X_test - X_mean) / X_std
    return X_train_norm, X_test_norm

In [9]:
def get_transformed_qubit_labels(y, qubit):

    # Ensure the qubit_position is between 1 and 5
    if qubit < 1 or qubit > 5:
        raise ValueError("qubit_position should be between 1 and 5.")
    
    # Adjust the starting index based on the reverse order: Q5 -> 0, Q4 -> 1, ..., Q1 -> 4

    y_binary_repr_original = np.array([f"{label:05b}" for label in y])
    y_binary_qubit_specific = np.array([1 if label[5 - qubit] == '1' else 0 for label in y_binary_repr_original])

    return y_binary_repr_original, y_binary_qubit_specific

In [10]:
def multiplexed_calculate_mf_rmf_envelopes(X, y, qubit):

    envelope_components = {}

    X_qubit = X 
    y_binary_repr_original, y_binary_qubit_specific = get_transformed_qubit_labels(y, qubit)

    # X_qubit, y_binary_repr_original, y_binary_qubit_specific = get_qubit_specific_data(X, y, qubit)
    print(f"Qubit: {qubit}, X_qubit.shape:{X_qubit.shape}, y_binary_repr_original.shape: {y_binary_repr_original.shape}")
    
    ground_state_traces, excited_state_traces, ground_state_labels, excited_state_labels = find_ground_excited_pairs(
        X_qubit, y_binary_repr_original, qubit)
    print(f"Qubit: {qubit}, Tr0.shape:{ground_state_traces.shape}, Tr1.shape: {excited_state_traces.shape}")

    relaxation_traces, relaxation_ground_state_traces = get_relaxation_traces(ground_state_traces, excited_state_traces)
    
    print(f'Qubit: {qubit}, Tr1Relaxed: {len(relaxation_traces)} perc: {(relaxation_traces.shape[0]/excited_state_traces.shape[0])*100}%')

    rmf_envelope_I, rmf_envelope_Q = compute_rmf_envelopes(relaxation_ground_state_traces, relaxation_traces)
    mf_envelope_I, mf_envelope_Q = compute_mf_envelopes(ground_state_traces, excited_state_traces)
    
    envelope_components[f'MF_I'] = mf_envelope_I
    envelope_components[f'MF_Q'] = mf_envelope_Q
    envelope_components[f'RMF_I'] = rmf_envelope_I
    envelope_components[f'RMF_Q'] = rmf_envelope_Q

    return X_qubit, y_binary_qubit_specific, envelope_components

### Load Data and Compute MF-RMF Envelopes

In [21]:
import pickle as pkl

In [22]:
data_path = 'five_qubit_data'
data_file_name = 'DRaw_C_Tr_v0-001'

In [27]:
mf_rmf_envelope_save_path = './single_qubit_data'

In [28]:
X, y = hdf5_data_load(data_path, data_file_name, data_type='Train')
X = X[:,:500,:]

In [29]:
qubits = [5, 4, 3, 2, 1]

In [30]:
for qubit in qubits:
    X_qubit, y_qubit_01, envelope_components = multiplexed_calculate_mf_rmf_envelopes(X, y, qubit)
    
    with open(f"{mf_rmf_envelope_save_path}/MF_RMF_q{qubit}.pkl", "wb") as f:
        pkl.dump(envelope_components, f)
    

Qubit: 5, X_qubit.shape:(480000, 500, 2), y_binary_repr_original.shape: (480000,)
Qubit: 5, Tr0.shape:(240000, 500, 2), Tr1.shape: (240000, 500, 2)
Qubit: 5, Tr1Relaxed: 706 perc: 0.2941666666666667%
Qubit: 4, X_qubit.shape:(480000, 500, 2), y_binary_repr_original.shape: (480000,)
Qubit: 4, Tr0.shape:(240000, 500, 2), Tr1.shape: (240000, 500, 2)
Qubit: 4, Tr1Relaxed: 4605 perc: 1.91875%
Qubit: 3, X_qubit.shape:(480000, 500, 2), y_binary_repr_original.shape: (480000,)
Qubit: 3, Tr0.shape:(240000, 500, 2), Tr1.shape: (240000, 500, 2)
Qubit: 3, Tr1Relaxed: 18247 perc: 7.602916666666666%
Qubit: 2, X_qubit.shape:(480000, 500, 2), y_binary_repr_original.shape: (480000,)
Qubit: 2, Tr0.shape:(240000, 500, 2), Tr1.shape: (240000, 500, 2)
Qubit: 2, Tr1Relaxed: 1702 perc: 0.7091666666666666%
Qubit: 1, X_qubit.shape:(480000, 500, 2), y_binary_repr_original.shape: (480000,)
Qubit: 1, Tr0.shape:(240000, 500, 2), Tr1.shape: (240000, 500, 2)
Qubit: 1, Tr1Relaxed: 162 perc: 0.0675%
